# RNA3D — trRosettaRNA2: 10-target single-sequence inference (Colab GPU)

**Goal:** Extend the frozen 10-target exploratory benchmark from three to four models. This notebook performs **inference only**, not evaluation, verification or manuscript writing.

**Prerequisites:** Google Colab with **T4 GPU**, accessible personal Google Drive `MyDrive/RNA3D_Benchmark`, and internet. Model release/commit pinned to the previously smoke-tested `d40a40e36617cb4004a11b0a7603167415ed9c52`. Outputs go to a **new** run folder; no existing run is overwritten. No GitHub writes.

**Conditions:** A3M inputs are the frozen `NuFold/runs/pilot10_single_20261018T203832Z/inputs/<target>/<target>.a3m` (single-sequence baseline), **not** genuine homologous MSA. Internal trRNA2-SS is used; native 3D structures never supplied. Genuine-MSA inference is a separate future ablation.

**Runtime note:** Installation can take substantial time. If you already used trRosettaRNA2 in this *same* active Colab runtime, bootstrap detects the existing environment and skips recreation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, subprocess, hashlib, csv, shutil, time, re
ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark')
assert ROOT.is_dir(), f'Missing Drive benchmark root: {ROOT}'
TARGETS = ['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
SOURCE = ROOT/'NuFold/runs/pilot10_single_20261018T203832Z/inputs'
RUN_ID = 'trrosettarna2_pilot10_single_20261114_v1'
DEST = ROOT/'trRosettaRNA2/runs'/RUN_ID
print('Source:', SOURCE, '\nOutput:', DEST)

## 1. Input preflight — must pass before installation

In [ ]:
def query_from_a3m(path):
    s=[]; found=False
    for line in path.read_text().splitlines():
        line=line.strip()
        if line.startswith('>'):
            if found: break
            found=True
        elif found: s.append(line)
    assert found, f'No A3M header: {path}'
    return ''.join(x for x in ''.join(s) if x not in '-.' and not x.islower()).upper().replace('T','U')
def seq_from_fasta(path):
    return ''.join(l.strip() for l in path.read_text().splitlines() if l.strip() and not l.startswith('>')).upper().replace('T','U')
INPUTS={}
for tid in TARGETS:
    a=SOURCE/tid/f'{tid}.a3m'; f=SOURCE/tid/f'{tid}.fasta'
    assert a.is_file() and f.is_file(), f'Missing {tid}: {a} / {f}'
    seq=query_from_a3m(a); fasta=seq_from_fasta(f)
    assert seq==fasta and set(seq)<=set('ACGU'), f'Sequence mismatch/invalid bases: {tid}'
    # frozen single-record A3M condition
    nseq=sum(line.startswith('>') for line in a.read_text().splitlines())
    assert nseq==1, f'{tid} unexpectedly has A3M depth {nseq}; stop to avoid mixing input conditions'
    INPUTS[tid]={'a3m':a,'fasta':f,'sequence':seq,'length':len(seq),'depth':nseq}
    print(f'{tid:10} {len(seq):3} nt  A3M depth={nseq}')
assert len(INPUTS)==10
print('INPUT PREFLIGHT PASS: 10/10')

## 2. Model bootstrap (GPU environment + weights)
Official upstream usage: `python -m trRNA2.predict -i INPUT.a3m -o OUTPUT`. The RNAcentral ~32GB database is **not** downloaded because A3Ms already exist.

In [ ]:
assert subprocess.run(['bash','-lc','nvidia-smi'],capture_output=True).returncode==0, 'Enable Colab GPU runtime first!'
REPO=Path('/content/trRosettaRNA2'); PIN='d40a40e36617cb4004a11b0a7603167415ed9c52'
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','https://github.com/YangLab-SDU/trRosettaRNA2.git',str(REPO)],check=True)
actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if actual!=PIN:
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',PIN],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==PIN
print('Pinned model commit:',PIN)

In [ ]:
# Install micromamba only if absent. The pinned upstream environment.yml is authoritative.
import urllib.request, tarfile, io
MM=Path('/content/bin/micromamba')
if not MM.exists():
    MM.parent.mkdir(parents=True,exist_ok=True)
    url='https://micro.mamba.pm/api/micromamba/linux-64/latest'
    print('Downloading micromamba...')
    blob=urllib.request.urlopen(url,timeout=120).read()
    with tarfile.open(fileobj=io.BytesIO(blob),mode='r:bz2') as tf:
        obj=tf.extractfile('bin/micromamba'); assert obj
        MM.write_bytes(obj.read())
    MM.chmod(0o755)
ENV_ROOT='/content/micromamba_root'
PYTHON=Path(ENV_ROOT+'/envs/trRNA2/bin/python')
if not PYTHON.exists():
    print('Creating pinned model environment (could take time)...')
    subprocess.run([str(MM),'-r',ENV_ROOT,'create','-y','-n','trRNA2','-f','environment.yml'],cwd=REPO,check=True)
assert PYTHON.exists(),f'Environment python missing: {PYTHON}'
p=subprocess.run([str(PYTHON),'-c','import torch; print(torch.__version__, torch.cuda.is_available())'],capture_output=True,text=True)
print(p.stdout,p.stderr[-1000:]); assert p.returncode==0 and 'True' in p.stdout, 'GPU torch check failed'
print('ENVIRONMENT PASS')

In [ ]:
# Download ONLY model weights if not already installed. Do not download the optional RNAcentral sequence DB.
import urllib.request
PARAM=REPO/'params'
if not PARAM.exists() or not any(PARAM.rglob('*.pt')) and not any(PARAM.rglob('*.pth')):
    ARCHIVE=REPO/'params_trRNA2.tar.bz2'
    if not ARCHIVE.is_file():
        urls=[
          'https://huggingface.co/datasets/quailwwk/trRNA2/resolve/main/params_trRNA2.tar.bz2',
          'http://yanglab.qd.sdu.edu.cn/trRosettaRNA/download/params_trRNA2.tar.bz2']
        last=None
        for url in urls:
            try:
                print('Downloading model weights from',url)
                urllib.request.urlretrieve(url,ARCHIVE)
                break
            except Exception as e: last=e; print(type(e).__name__,str(e)[:250])
        else: raise RuntimeError(f'Both model weight mirrors failed: {last}')
    subprocess.run(['tar','-xjf',str(ARCHIVE)],cwd=REPO,check=True)
print('Model weights downloaded / already present. Check upstream filenames before inference if preflight fails.')

## 3. Resume-safe 10-target inference
Runs each target only once, records errors and continues to the next. The final successful PDB (`model_1_relaxed200.pdb`) is copied into a new per-target Drive folder. Restarting this exact notebook skips already complete targets only if the saved input and output hashes match.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def pdb_residue_count(path):
    seen=set()
    for l in Path(path).read_text(errors='replace').splitlines():
        if l.startswith(('ATOM  ','HETATM')) and len(l)>=27 and l[16] in ' A1':
            seen.add((l[21],l[22:26],l[26]))
    return len(seen)

DEST.mkdir(parents=True,exist_ok=True)
results=[]
for tid in TARGETS:
    info=INPUTS[tid]; out=DEST/tid
    out.mkdir(exist_ok=True)
    pred=out/'model_1_relaxed200.pdb'; record=out/'inference.json'
    a3mhash=sha256(info['a3m'])
    if pred.exists() and record.exists():
        old=json.loads(record.read_text())
        assert old['input_a3m_sha256']==a3mhash, f'Input changed in existing run for {tid}'
        assert old.get('output_pdb_sha256')==sha256(pred), f'Output changed in existing run for {tid}'
        assert old.get('status')=='PASS', f'Incomplete output for {tid}'
        print('SKIP VERIFIED',tid)
        results.append(old);continue
    if pred.exists() or record.exists():
        raise RuntimeError(f'Incomplete pre-existing run for {tid}; refusing to overwrite: {out}')
    # Snapshot source A3M and FASTA, never overwrite different data.
    for key in ('a3m','fasta'):
        src=info[key]; dst=out/src.name
        if dst.exists():assert sha256(dst)==sha256(src)
        else:shutil.copy2(src,dst)
    scratch=Path('/content')/'trrna2_scratch'/tid
    if scratch.exists():shutil.rmtree(scratch)
    scratch.mkdir(parents=True)
    cmd=[str(PYTHON),'-m','trRNA2.predict','-i',str(out/f'{tid}.a3m'),'-o',str(scratch)]
    print('RUN',tid, ' '.join(cmd))
    t0=time.monotonic()
    p=subprocess.run(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    elapsed=round(time.monotonic()-t0,3)
    (out/'inference.log').write_text(p.stdout)
    produced=scratch/'model_1_relaxed200.pdb'
    status='PASS' if p.returncode==0 and produced.exists() and produced.stat().st_size>0 else 'FAIL'
    rec=dict(target_id=tid,model='trRosettaRNA2',model_commit=PIN,input_mode='single_sequence_a3m',
             internal_secondary_structure=True,input_a3m_sha256=a3mhash,sequence_length=info['length'],
             msa_depth=info['depth'],runtime_seconds=elapsed,exit_code=p.returncode,status=status,
             output_pdb_sha256=None,output_residues=None)
    if status=='PASS':
        rec['output_residues']=pdb_residue_count(produced)
        if rec['output_residues']!=info['length']:
            status=rec['status']='FAIL_RESIDUE_COUNT'
        else:
            shutil.copy2(produced,pred)
            rec['output_pdb_sha256']=sha256(pred)
    record.write_text(json.dumps(rec,indent=2)+'\n')
    results.append(rec)
    print(tid,status,'seconds=',elapsed,'residues=',rec['output_residues'])
    if status!='PASS': print('LOG TAIL:',p.stdout[-2500:])

print('Completed',sum(x['status']=='PASS' for x in results),'of',len(TARGETS))

## 4. Summary manifest (do not misreport failures as success)

In [ ]:
with (DEST/'batch_summary.csv').open('w',newline='') as f:
    fields=['target_id','model','model_commit','input_mode','sequence_length','msa_depth','runtime_seconds','exit_code','status','output_residues','input_a3m_sha256','output_pdb_sha256']
    w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');w.writeheader();w.writerows(results)
passcount=sum(x['status']=='PASS' for x in results)
(DEST/'run_status.json').write_text(json.dumps({'run_id':RUN_ID,'status':'complete' if passcount==10 else 'incomplete',
          'passed':passcount,'targets':10,'model_commit':PIN,'mode':'single-sequence A3M',
          'native_used_for_inference':False,'github_writes':False},indent=2)+'\n')
print('Final:',f'{passcount}/10 PASS', 'folder:',DEST)
assert passcount==10, f'{10-passcount} failures. Inspect corresponding inference.log before scoring.'
print('TRROSETTARNA2 SINGLE-SEQUENCE 10/10 COMPLETE — READY FOR EVALUATION')